# 22 · Fixing the three defects the manual review found

**Purpose.** The manual review (error-analysis-2 step 4, D57) named three concrete defects in the
hand-set X2 score (`2·role + experience + 0.999·TF-IDF`). This notebook shows what fixing each one
does, so the owner can decide whether to implement any of them in the pipeline.

| ID | Defect | Fix tried here |
|---|---|---|
| F1 | A skill in a negated clause ("No Angular/Vue projects") still counts | drop clauses with a negation cue and a usage word from the CV text, then recompute every term |
| F2 | A CV with no work history gets an unknown experience band, which X2 counts as compatible | experience = 0 when the CV says it has no professional experience **and** the JD asks for years |
| F2b | (the blunt version of F2) | every unknown experience band = 0 (the rung-1a rule) |
| F3 | The role model calls a same-role pair out of role | diagnostic only: a role-model change is training, its own plan |

**Status.** `exploratory`. The owner deferred the prevalence floor and the bar until after reading
this notebook, so nothing here is a registered test (rule 8). Nothing is fitted; A2 final is scored
on, never tuned on (D46). Every figure is read from `output/extraction/score-defects.json`
(`python -m candidate_screener.evaluation.score_defects`); this notebook defines no model.

**Where it runs.** D50 selection (70 JDs, where the rules were written), D50 held-out (30 JDs), and
A2 final (59 JDs with both classes, by stratum). D50 labels are LLM-only; A2 final is the reference
(D41). AUC is the mean per-JD AUC, Fit vs No Fit (D47); CIs are 95%, resampled over JDs.

In [1]:
import json

import pandas as pd

from candidate_screener.config import OUTPUT

r = json.load(open(OUTPUT / "extraction" / "score-defects.json"))
SETS = ["D50 selection (70 JDs)", "D50 held-out (30 JDs)", "A2 final (59 JDs)"]
ARMS = ["F1 negation", "F2 no experience", "F2b unknown = 0", "F1 + F2"]

## 1 · How often each defect occurs (prevalence)

*Pairs whose score changes* counts any change, however small: F1 edits the CV text, which nudges the
per-JD-scaled TF-IDF of many pairs without reordering them.

In [2]:
pd.DataFrame({s: r["sets"][s]["prevalence"] for s in SETS}).mul(100).round(1).rename_axis("% of CVs or pairs")

,D50 selection (70 JDs),D50 held-out (30 JDs),A2 final (59 JDs)
% of CVs or pairs,,,
CVs stating no professional experience,0.3,0.3,0.5
CVs with a negated clause,5.9,10.7,6.7
pairs whose F1 score changes,17.0,30.3,23.0
pairs whose F2 score changes,0.3,0.3,0.3
pairs whose F2b score changes,70.7,73.3,70.8
pairs with an unknown experience band,70.7,73.3,70.8


## 2 · Mean per-JD AUC of each version

In [3]:
rows = {}
for s in SETS:
    for scope, _ in r["sets"][s]["auc"]["X2"].items():
        rows[(s, scope)] = {a: r["sets"][s]["auc"][a][scope] for a in ["X2"] + ARMS}
pd.DataFrame(rows).T.round(3)

X2  F1 negation  F2 no experience  \
D50 selection (70 JDs) total     0.869        0.869             0.870   
D50 held-out (30 JDs)  total     0.859        0.855             0.863   
A2 final (59 JDs)      generic   0.863        0.858             0.866   
                       targeted  0.826        0.826             0.826   
                       total     0.850        0.847             0.853   

                                 F2b unknown = 0  F1 + F2  
D50 selection (70 JDs) total               0.852    0.869  
D50 held-out (30 JDs)  total               0.851    0.859  
A2 final (59 JDs)      generic             0.853    0.862  
                       targeted            0.786    0.826  
                       total               0.830    0.850

## 3 · Paired change against X2

Δ = mean over JDs of (AUC with the fix − AUC of X2), on the same JDs. *JDs up / down* counts the JDs
whose AUC moved; every other JD is unchanged. For A2, `generic`, `targeted` and `total` are shown
together (rule 3).

In [4]:
rows = {}
for s in SETS:
    for a in ARMS:
        for scope, d in r["sets"][s]["delta_vs_x2"][a].items():
            rows[(s, a, scope)] = {"Δ AUC": round(d["delta"], 4), "95% CI": str([round(c, 4) for c in d["ci"]]),
                                   "JDs up": d["jds_up"], "JDs down": d["jds_down"], "n JDs": d["n_jd"]}
pd.DataFrame(rows).T

Δ AUC              95% CI  \
D50 selection (70 JDs) F1 negation      total     0.0002   [-0.0021, 0.0028]   
                       F2 no experience total     0.0009       [0.0, 0.0028]   
                       F2b unknown = 0  total    -0.0169   [-0.0373, 0.0028]   
                       F1 + F2          total     0.0002   [-0.0021, 0.0028]   
D50 held-out (30 JDs)  F1 negation      total    -0.0045      [-0.0134, 0.0]   
                       F2 no experience total     0.0043       [0.0, 0.0129]   
                       F2b unknown = 0  total    -0.0085   [-0.0422, 0.0231]   
                       F1 + F2          total    -0.0002   [-0.0134, 0.0129]   
A2 final (59 JDs)      F1 negation      generic  -0.0045   [-0.0171, 0.0037]   
                                        targeted     0.0          [0.0, 0.0]   
                                        total     -0.003   [-0.0113, 0.0024]   
                       F2 no experience generic   0.0032       [0.0, 0.0096]   
                                        targeted     0.0          [0.0, 0.0]   
                                        total     0.0021       [0.0, 0.0064]   
                       F2b unknown = 0  generic  -0.0095     [-0.043, 0.028]   
                                        targeted -0.0405  [-0.0811, -0.0025]   
                                        total      -0.02   [-0.0469, 0.0076]   
                       F1 + F2          generic  -0.0013   [-0.0159, 0.0101]   
                                        targeted     0.0          [0.0, 0.0]   
                                        total    -0.0008   [-0.0105, 0.0067]   

                                                 JDs up JDs down n JDs  
D50 selection (70 JDs) F1 negation      total         1        1    68  
                       F2 no experience total         1        0    68  
                       F2b unknown = 0  total        11       19    68  
                       F1 + F2          total         1        1    68  
D50 held-out (30 JDs)  F1 negation      total         0        1    28  
                       F2 no experience total         1        0    28  
                       F2b unknown = 0  total         5        5    28  
                       F1 + F2          total         1        1    28  
A2 final (59 JDs)      F1 negation      generic       1        1    39  
                                        targeted      0        0    20  
                                        total         1        1    59  
                       F2 no experience generic       1        0    39  
                                        targeted      0        0    20  
                                        total         1        0    59  
                       F2b unknown = 0  generic       6       10    39  
                                        targeted      3        8    20  
                                        total         9       18    59  
                       F1 + F2          generic       2        1    39  
                                        targeted      0        0    20  
                                        total         2        1    59

## 4 · F3: where the role model sits on the reviewed cases

From the 45 reviewed mistakes (`error-review-reasons.csv`). Tag A = "role mismatch, but the Fit CV
has transferable skills"; the reviewer's notes on these say the JD and Fit CV are the same role.

In [5]:
pd.Series(r["f3_role_misses"], name="reviewed cases").to_frame()

,reviewed cases
A: Fit CV out of role (role < 0.5),6.0000
"A: median role_match, Fit side",0.0385
all reviewed: Fit side out of role,30.0000
all reviewed: No Fit side out of role,25.0000
n_reviewed,45.0000
n_tagged_A,6.0000


## 5 · Reading

1. **F1 (negation) does not help.** Negated clauses are in 6-11% of CVs and the fix nudges 17-30% of
   pair scores, but it reorders pairs in only one or two JDs per set, and the net change is about 0
   (A2 total −0.003, CI [−0.011, 0.002]). The rule is also imprecise: a spot check of D50 selection
   clauses found real negated skills ("without experience in web development") next to preferences
   ("not interested in maintenance projects") and false hits ("worked without designs").
   The Angular case from the review is caught, but it is one CV.
2. **F2 (stated no experience) is right but rare.** About 0.3-0.5% of CVs say they have no
   professional experience; the fix moves one JD up per set and none down (A2 total +0.002). It
   cannot reach the bar of any size that matters at this prevalence.
3. **F2b (unknown = 0) hurts.** About 71-73% of pairs have an unknown experience band, so zeroing
   them punishes most CVs for missing data, not missing experience: A2 total −0.020, targeted
   −0.041 with a CI below 0. This confirms X1's registered choice to count unknown as compatible.
4. **F3 is not a quick fix.** 30 of the 45 reviewed Fit CVs are out of role by the model; the six
   tagged A have a median role score of 0.04, so the model is confident, not borderline. Fixing it
   means retraining the role-family model.

**Overall.** None of the cheap fixes moves the score. Together with error-analysis-2 (no lever
reaches 0.02, label noise the largest reason in the review), the remaining error is mostly not
in these defects.

## 6 · For the owner to decide

- **Prevalence floor and bar** (deferred): at the figures above, F1 and F2 fail any positive bar.
  If you still want F2 in the pipeline, it is harmless (it never moved a JD down) and correct by
  reading.
- **Parked idea:** raising the experience and coverage weights in X2. Not run here; it is a separate
  exploratory arm, designed on D50 selection and confirmed on A2 final.